# Classificazione binaria

> **Nota AI Lab — ambiente di lettura.** Il testo seguente è la traduzione del notebook Kaggle originale conservato nel repository. I riferimenti a pulsanti, celle nascoste e invii descrivono l’ambiente Kaggle dell’originale; qui le celle di codice sono visibili. Per eseguire i notebook fuori da Kaggle, segui la guida AI Lab all’ambiente.



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione

Finora abbiamo imparato come le reti neurali possono risolvere problemi di regressione. Ora le applicheremo a un altro problema comune del machine learning: la classificazione. Quasi tutto ciò che abbiamo imparato continua a essere valido. La differenza principale riguarda la funzione di perdita e il tipo di uscita che vogliamo ottenere dallo strato finale.

# Classificazione binaria

Assegnare un elemento a una di due classi è un problema comune di machine learning. Potresti voler prevedere se un cliente probabilmente effettuerà un acquisto, se una transazione con carta di credito è fraudolenta, se segnali provenienti dallo spazio profondo indicano un nuovo pianeta, oppure se un esame medico mostra indizi di una malattia. Sono tutti problemi di **classificazione binaria**.

Nei dati grezzi le classi potrebbero essere rappresentate da stringhe come `"Yes"` e `"No"`, oppure `"Dog"` e `"Cat"`. Prima di usare i dati assegneremo un’**etichetta di classe**: una classe sarà `0` e l’altra `1`. Le etichette numeriche mettono i dati in una forma utilizzabile dalla rete neurale.

# Accuracy e cross-entropy

L’**accuracy** è una delle molte metriche usate per misurare il successo nella classificazione. È il rapporto tra previsioni corrette e previsioni totali: `accuracy = number_correct / total`. Un modello sempre corretto avrebbe un’accuracy di `1.0`. A parità di altre condizioni, è una metrica ragionevole quando le classi nel dataset hanno frequenze approssimativamente uguali.

Il problema dell’accuracy, come della maggior parte delle metriche di classificazione, è che non può essere usata come funzione di perdita. SGD richiede una funzione che vari in modo regolare, mentre l’accuracy, essendo un rapporto tra conteggi, cambia «a salti». Dobbiamo quindi scegliere un sostituto come funzione di perdita: la *cross-entropy*, o entropia incrociata.

Ricorda che la funzione di perdita definisce l’*obiettivo* della rete durante il training. Nella regressione volevamo minimizzare la distanza tra risultato atteso e risultato previsto. Per misurarla avevamo scelto il MAE.

Nella classificazione vogliamo invece una distanza tra *probabilità*, ed è ciò che fornisce la cross-entropy. La **cross-entropy** è una sorta di misura della distanza tra una distribuzione di probabilità e un’altra.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/DwVV9bR.png" width="400" alt="Grafici dell’accuracy e della cross-entropy.">
<figcaption><center>La cross-entropy penalizza le previsioni di probabilità scorrette.</center></figcaption>
</figure>

L’idea è che vogliamo che la rete preveda la classe corretta con probabilità `1.0`. Più la probabilità prevista si allontana da `1.0`, maggiore sarà la perdita cross-entropy.

Le ragioni tecniche per usare la cross-entropy sono un po’ sottili, ma il punto principale è questo: usa la cross-entropy come perdita per la classificazione; le altre metriche che ti interessano, come l’accuracy, tenderanno a migliorare insieme a essa.

# Produrre probabilità con la funzione sigmoid

Cross-entropy e accuracy richiedono entrambe probabilità in ingresso, cioè numeri tra 0 e 1. Per convertire in probabilità le uscite a valori reali di uno strato dense, aggiungiamo un nuovo tipo di funzione di attivazione: l’**attivazione sigmoid**.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/FYbRvJo.png" width="400" alt="Grafico a S della sigmoid, con asintoti orizzontali a 0 a sinistra e a 1 a destra.">
<figcaption><center>La funzione sigmoid associa i numeri reali all’intervallo $[0, 1]$.</center></figcaption>
</figure>

Per ottenere la previsione finale della classe, definiamo una probabilità di *soglia*. In genere è 0,5: arrotondando otteniamo la classe, con valori sotto 0,5 associati all’etichetta 0 e valori uguali o superiori a 0,5 associati all’etichetta 1. La soglia 0,5 è quella usata da Keras per impostazione predefinita nella sua [metrica di accuracy](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/BinaryAccuracy).

# Esempio: classificazione binaria

Ora proviamoci!

Il dataset [Ionosphere](https://archive.ics.uci.edu/ml/datasets/Ionosphere) contiene feature ricavate da segnali radar diretti verso la ionosfera terrestre. Il compito è stabilire se il segnale mostri la presenza di un oggetto oppure soltanto aria vuota.


In [ ]:

import pandas as pd
from IPython.display import display

ion = pd.read_csv('../input/dl-course-data/ion.csv', index_col=0)
display(ion.head())

df = ion.copy()
df['Class'] = df['Class'].map({'good': 0, 'bad': 1})

df_train = df.sample(frac=0.7, random_state=0)
df_valid = df.drop(df_train.index)

max_ = df_train.max(axis=0)
min_ = df_train.min(axis=0)

df_train = (df_train - min_) / (max_ - min_)
df_valid = (df_valid - min_) / (max_ - min_)
df_train.dropna(axis=1, inplace=True) # Rimuovi la feature vuota nella colonna 2.
df_valid.dropna(axis=1, inplace=True)

X_train = df_train.drop('Class', axis=1)
X_valid = df_valid.drop('Class', axis=1)
y_train = df_train['Class']
y_valid = df_valid['Class']

Definiremo il modello come per i compiti di regressione, con una sola eccezione: nello strato finale includeremo un’attivazione `'sigmoid'`, così il modello produrrà probabilità di classe.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(4, activation='relu', input_shape=[33]),
    layers.Dense(4, activation='relu'),    
    layers.Dense(1, activation='sigmoid'),
])

Aggiungi la perdita cross-entropy e la metrica accuracy al modello usando `compile`. Per i problemi a due classi assicurati di usare le versioni `'binary'`; i problemi con più classi sono leggermente diversi. Adam funziona molto bene anche nella classificazione, quindi continueremo a usarlo.


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

In questo problema il modello può richiedere parecchie epoche per completare l’addestramento: per comodità includeremo quindi una callback di early stopping.


In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=10,
    min_delta=0.001,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=1000,
    callbacks=[early_stopping],
    verbose=0, # Nascondi l'output perché le epoche sono numerose.
)

Come sempre, osserveremo le curve di apprendimento ed esamineremo anche i migliori valori di perdita e accuracy ottenuti sul validation set. Ricorda che l’early stopping ripristinerà i pesi che hanno prodotto questi valori.


In [ ]:
history_df = pd.DataFrame(history.history)
# Inizia il grafico dall'epoca 5.
history_df.loc[5:, ['loss', 'val_loss']].plot()
history_df.loc[5:, ['binary_accuracy', 'val_binary_accuracy']].plot()

print(("Best Validation Loss: {:0.4f}" +\
      "\nBest Validation Accuracy: {:0.4f}")\
      .format(history_df['val_loss'].min(), 
              history_df['val_binary_accuracy'].max()))

# Ora tocca a te

Usa una rete neurale per [**prevedere le cancellazioni delle prenotazioni alberghiere**](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex6.html) con il dataset *Hotel Cancellations*.
